In [15]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Make Sure inline plotting is activated for jupyter Notebooks
%matplotlib inline
# ===============================================================
# 1. INGESTION & INTERACTIVE AUDIT
# ===============================================================
df = pd.read_csv('raw_loan_portfolio.csv')

print("=== STEP:1 INITIAL META DATA AUDIT===")
print(f"Total Rows Loaded {df.shape[0]} | Columns:{df.shape[1]}")
print("\n---Structural Null Counts---")
print(df.isnull().sum())

# In Jupyter, we can use .head() at the end of a cell to see a clean visual table layout
# For now, we print it to audit the initial raw text format
print("\n --- Raw Data Snippet---")
print(df[['Requested_Amount_Raw', 'Internal_Risk_Grade']].head(3))
print("===========================================\n")

# =================================================================
# 2. STRING CLEANING & CURRENCY STRING TO NUMERIC FLOAT CASTING
# =================================================================
# Clean the dirty currency string (' $25,000.00 ') so we can perform math operations

df['Loan_Amount_USD'] = df['Requested_Amount_Raw'].str.strip()
df['Loan_Amount_USD']= df['Loan_Amount_USD'].str.replace('$','',regex=False)
df['Loan_Amount_USD']= df['Loan_Amount_USD'].str.replace(',','',regex=False)
df['Loan_Amount_USD'] = df['Loan_Amount_USD'].astype(float)

# Standardize text anomalies across categorical variables
df['Internal_Risk_Grade'] = df['Internal_Risk_Grade'].str.strip().str.upper()
courrupt_markers = {'UNKNOWN':np.nan, 'N/A': np.nan}
df['Internal_Risk_Grade'] = df['Internal_Risk_Grade'].replace(courrupt_markers)
df['Loan_Purpose'] = df['Loan_Purpose'].replace(courrupt_markers)

# Drop rows missing our core target category label
df = df.dropna(subset=['Internal_Risk_Grade'])

# =================================================================
# 3. ADVANCED BOOLEAN RISK FILTERING & LOGICAL MASKS
# =================================================================
# Exclusive impossible negative debt values and system glitch values (99.9)
valid_ratio_mask = (df['Customer_Debt_To_Income_Ratio'] >= 0) & (df['Customer_Debt_To_Income_Ratio'] <= 70)
df = df[valid_ratio_mask]

# Create a critical custom alert feature: High levarage risk Accounts (Debt > 40% AND Loan > $30k)
df['High_Risk_Alert'] = np.where((df['Customer_Debt_To_Income_Ratio'] > 40) & (df['Loan_Amount_USD'] > 30000), 1,0)

# ==================================================================
# 4. PORTFOLIO RISK AGGREGATION & STORAGE
# ==================================================================

credit_risk_summary = df.groupby(['Internal_Risk_Grade','Loan_Purpose']).agg(
    Active_Accounts = ('Account_ID','count'),
    Total_Capital_Exposed_USD= ('Loan_Amount_USD','sum'),
    Average_Debt_Ratio = ('Customer_Debt_To_Income_Ratio','mean'),
    Default_Rate_Percentage = ('Loan_Default_Status',lambda x:x.mean() * 100),
    Trigger_Risk_Alerts = ('High_Risk_Alert','sum')
).reset_index()

credit_risk_summary.to_csv('cleaned_credit_risk_summary.csv', index =False)
print("📥 Processed summary table compiled and saved as credit_risk_summary.to_csv to the folder.\n")

=== STEP:1 INITIAL META DATA AUDIT===
Total Rows Loaded 7000 | Columns:6

---Structural Null Counts---
Account_ID                         0
Internal_Risk_Grade              364
Loan_Purpose                     385
Customer_Debt_To_Income_Ratio    941
Loan_Default_Status                0
Requested_Amount_Raw               0
dtype: int64

 --- Raw Data Snippet---
  Requested_Amount_Raw Internal_Risk_Grade
0          $28,393.00              Class-A
1          $44,347.00                  NaN
2          $44,149.00              Class-B

📥 Processed summary table compiled and saved as credit_risk_summary.to_csv to the folder.

